# TP  Représentation du texte pour la classification de sentiments

  <table style="margin: 0 auto; color: white; font-size: 0.95em;">
    <tr><td style="padding:4px 20px;"><b>Établissement</b></td><td>ENSAM — Université Hassan II de Casablanca</td></tr>
    <tr><td style="padding:4px 20px;"><b>Année</b></td><td>2025 – 2026</td></tr>
    <tr><td style="padding:4px 20px;"><b>Cours</b></td><td>NLP — Traitement du Langage Naturel</td></tr>
    <tr><td style="padding:4px 20px;"><b>Dataset fil rouge</b></td><td>IMDb Movie Reviews (50 000 critiques)</td></tr>
    <tr><td style="padding:4px 20px;"><b>Tâche</b></td><td>Classification de sentiment : Positif / Négatif</td></tr>
  </table>

Ce notebook propose une mise en pratique progressive autour d’un problème classique de NLP : **classer des avis positifs ou négatifs à partir de leur texte**.

L’objectif pédagogique est double :
1. comprendre la logique des représentations textuelles ;
2. comparer, sur un même corpus, des approches de complexité croissante.

Vous pouvez travailler sur :
- **IMDb** (`IMDB Dataset of 50K Movie Reviews`) ;
    - https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

- ou un **corpus personnel** en classification supervisée.

Le travail suit une progression expérimentale :
**exploration → préparation → vectorisation → classification → évaluation → interprétation**.

### Conseils méthodologiques

Quelques principes à garder:
- une bonne représentation dépend de la tâche ;
- une meilleure représentation n’est pas toujours plus interprétable ;
- les gains de performance doivent être mis en regard du coût de calcul ;
- les erreurs observées sont souvent plus instructives que le score final.

### Librairies nécessaires
| Librairie | Rôle |
|-----------|------|
| `scikit-learn` | TF-IDF, BoW, métriques d'évaluation |
| `gensim` | Word2Vec, GloVe, FastText |
| `transformers` | BERT et modèles HuggingFace |
| `torch` | Backend pour BERT |
| `datasets` | Chargement du dataset IMDb depuis HuggingFace |
| `matplotlib / seaborn` | Visualisations |

In [ ]:
# # Installation (décommenter et exécuter une seule fois) 
# !pip install scikit-learn gensim transformers torch datasets
# !pip install numpy pandas matplotlib seaborn umap-learn -q

In [ ]:
import re
import time
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.manifold import TSNE

random.seed(42)
np.random.seed(42)
sns.set_style("whitegrid")

## 1. Cadre expérimental

Avant toute modélisation, clarifiez le cadre de travail.

### Questions
- Quel corpus est utilisé ?
- Quelle est la variable cible ?
- Travaillez-vous sur l’ensemble du corpus ou sur un sous-échantillon ?
- Les classes sont-elles équilibrées ?

### Réponses
- **Corpus** : IMDb Movie Reviews (texte en anglais, sentiment binaire).
- **Variable cible** : `label` avec 0 = négatif, 1 = positif.
- **Volume de travail** : sous-échantillon équilibré de 12 000 exemples pour accélérer l’expérimentation.
- **Équilibre** : oui, le sous-échantillon est construit de manière équilibrée (50/50).

### Point méthodologique
Pour comparer plusieurs représentations, il est préférable de conserver **le même découpage apprentissage / test** pour toutes les expériences.

### Dataset IMDb Movie Reviews

#### Présentation du Dataset

Le dataset **IMDb Movie Reviews** est un référentiel standard du NLP pour l'analyse de sentiment.
Il contient **50 000 critiques de films en anglais**, équitablement réparties entre positif et négatif.

| Caractéristique | Valeur |
|-----------------|--------|
| Source | Internet Movie Database (IMDb) |
| Taille totale | 50 000 critiques |
| Entraînement | 25 000 (50% pos / 50% neg) |
| Test | 25 000 (50% pos / 50% neg) |
| Langue | Anglais |
| Tâche | Classification binaire : Positif (1) / Négatif (0) |
| Référence | Maas et al., 2011 — Stanford NLP |

#### Pourquoi ce dataset ?

Ce dataset est **idéal pour comparer les méthodes** car :
- Il contient de la **négation** : "not good", "not bad" → difficile pour BoW
- Il contient des **synonymes** : "excellent" ≈ "brilliant" → difficile pour TF-IDF
- Il contient des **mots rares et argotiques** → difficile pour Word2Vec
- Il nécessite une **compréhension du contexte** → BERT excelle ici
- Il est **bien équilibré** → métriques fiables

> **Pour faciliter l’expérimentation, vous pouvez travailler sur un sous‑ensemble du jeu de données (par exemple 10 000 à 20 000 exemples)**.

- Colonnes attendues :
    - text : le contenu textuel
    - label : la classe (0/1 ou autre encodage supervisé)

In [ ]:
from datasets import load_dataset

# Chargement IMDb depuis Hugging Face
imdb = load_dataset("imdb")
train_df = imdb["train"].to_pandas()
test_df = imdb["test"].to_pandas()

# Fusion train+test pour garder un protocole unique ensuite
full_df = pd.concat([train_df, test_df], ignore_index=True)
full_df = full_df[["text", "label"]].dropna().reset_index(drop=True)

# Sous-échantillon équilibré pour accélérer les expérimentations
N_SAMPLES = 12000
per_class = N_SAMPLES // 2
df_pos = full_df[full_df["label"] == 1].sample(n=per_class, random_state=42)
df_neg = full_df[full_df["label"] == 0].sample(n=per_class, random_state=42)
df = pd.concat([df_pos, df_neg], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Taille corpus total IMDb: {len(full_df):,}")
print(f"Taille sous-échantillon utilisé: {len(df):,}")
print(df["label"].value_counts())
df.head()

### Aide pour un corpus personnel

Si vous utilisez vos propres données, vérifiez que :
- chaque ligne correspond à un document ;
- le texte est bien dans une seule colonne ;
- l’étiquette de classe est disponible ;
- les valeurs manquantes ont été retirées ou traitées.

## 2. Exploration du corpus

### Objectifs
Observer la structure des données avant de les transformer.

> Une bonne exploration permet souvent d’anticiper les limites des représentations simples.

### Questions
- Combien de mots contiennent les avis, en moyenne ?
- Les textes sont-ils très hétérogènes en longueur ?
- Certaines classes utilisent-elles un vocabulaire plus “marqué” que d’autres ?
- Voyez-vous du bruit textuel utile à nettoyer ?

### Réponses
- Les avis contiennent en moyenne plusieurs centaines de mots (à confirmer par `describe()` exécuté).
- Oui, la longueur est hétérogène avec une distribution étalée et des extrêmes.
- Le vocabulaire est globalement similaire, mais certains termes sont fortement corrélés à la polarité.
- Oui, présence de balises HTML et de ponctuation/caractères non alphabétiques, d’où le nettoyage léger appliqué.

In [ ]:
df["length"] = df["text"].astype(str).str.split().apply(len)

display(df["length"].describe())
display(df.groupby("label")["length"].describe())

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df["length"], bins=40, color="#1f77b4", alpha=0.8)
plt.title("Distribution des longueurs de textes")
plt.xlabel("Nombre de mots")
plt.ylabel("Fréquence")
plt.show()

### Analyse à rédiger

**Observations principales (IMDb, sous-échantillon équilibré)**
- **Équilibre des classes** : le corpus utilisé est équilibré (50% positif / 50% négatif), ce qui rend l’accuracy et le F1 macro cohérents pour comparer les méthodes.
- **Longueur des textes** : les avis IMDb sont en moyenne assez longs, mais avec une forte dispersion (textes très courts et très longs coexistent).
- **Bruit textuel** : présence de balises HTML (`<br />`), ponctuation, variantes de casse et caractères spéciaux; un nettoyage léger améliore la stabilité des modèles lexicaux.
- **Sous-échantillonnage** : un sous-échantillon (ici 12 000) conserve les tendances globales tout en réduisant fortement le coût de calcul, surtout pour les embeddings et BERT.

## 3. Prétraitement

Le prétraitement doit être justifié. Dans une tâche de sentiment, certaines décisions peuvent être bénéfiques dans un contexte, et nuisibles dans un autre.

### Questions
- Faut-il supprimer les stopwords ?
- Faut-il conserver les négations comme `not`, `no`, `never` ?
- Faut-il conserver l’apostrophe et les contractions ?
- Faut-il normaliser l’écriture en minuscules ?

### Réponses retenues pour ce TP
- **Stopwords** : on ne les supprime pas ici, car certains mots fonctionnels participent au sens de polarité selon le contexte.
- **Négations** : on les conserve impérativement (`not`, `no`, `never`) car elles inversent souvent le sentiment.
- **Apostrophes / contractions** : on les conserve pour ne pas casser des formes comme `didn't`, `wasn't`.
- **Minuscules** : oui, pour réduire la sparsité (Good/good).

### Point d’attention

Évitez de retirer automatiquement les mots de négation sans réflexion.  
Dans l’analyse de sentiment, cette décision peut supprimer une information essentielle.

In [ ]:
def nettoyer_texte(texte: str) -> str:
    texte = str(texte).lower()
    texte = re.sub(r"<br\s*/?>", " ", texte)
    # On garde apostrophes et espaces pour préserver des contractions utiles au sentiment.
    texte = re.sub(r"[^a-zA-Z\s']", " ", texte)
    texte = re.sub(r"\s+", " ", texte).strip()
    return texte

df["text_clean"] = df["text"].apply(nettoyer_texte)
df[["text", "text_clean"]].head()

In [ ]:
df["label"].value_counts(normalize=True).sort_index().plot(kind="bar", color=["#d62728", "#2ca02c"])
plt.title("Répartition des classes")
plt.xticks([0, 1], ["Négatif (0)", "Positif (1)"], rotation=0)
plt.ylabel("Proportion")
plt.show()

## 4. Protocole de comparaison

Avant de tester les méthodes, mettez en place un cadre commun.

### À conserver constant
- le découpage train/test ;
- le classifieur de base ;
- les métriques ;
- le niveau de prétraitement.

### Pourquoi ?
Ainsi, les écarts observés proviennent principalement de la représentation textuelle, et non d’un changement de protocole.

In [ ]:
# Variables de travail
X = df["text_clean"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape, "| Test:", X_test.shape)

## 5.  Bag of Words

Le Bag of Words transforme chaque texte en vecteur de fréquences lexicales.

### Questions
- Quelles informations conserve cette représentation ?
- Quelles informations perd-elle ?
- Pourquoi peut-elle rester performante sur IMDb ?
- Quels types d’erreurs peut-elle produire ?

### Réponses
- BoW conserve surtout la présence et la fréquence des mots.
- BoW perd l’ordre, la syntaxe, les dépendances longues et une partie du contexte.
- Sur IMDb, beaucoup de mots sont fortement polarisés, ce qui suffit souvent à obtenir de bonnes performances.
- Erreurs typiques: négation mal interprétée, ironie, phrases contrastées (`good acting but terrible story`).

In [ ]:
bow_model = Pipeline([
    ("vect", CountVectorizer(max_features=10000)),
    ("clf", LogisticRegression(max_iter=1000))
])

t0 = time.time()
bow_model.fit(X_train, y_train)
time_bow = time.time() - t0

y_pred_bow = bow_model.predict(X_test)
acc_bow = accuracy_score(y_test, y_pred_bow)
f1_bow = f1_score(y_test, y_pred_bow, average="macro")

print("Accuracy:", round(acc_bow, 4))
print("F1 macro:", round(f1_bow, 4))
print("Temps (s):", round(time_bow, 2))

In [ ]:
vocab = bow_model.named_steps["vect"].get_feature_names_out()
coef = bow_model.named_steps["clf"].coef_[0]

top_pos = np.argsort(coef)[-10:][::-1]
top_neg = np.argsort(coef)[:10]

print("Mots les plus associés à la classe positive:")
print(vocab[top_pos])
print("\nMots les plus associés à la classe négative:")
print(vocab[top_neg])

### Interprétation guidée

Commentaires :
1. **Oui**, le modèle repère des mots sentimentaux explicites (`excellent`, `great`, `bad`, `worst`, etc.).
2. **Non**, BoW n’est pas sensible à l’ordre des mots; il travaille surtout sur la présence/fréquence des termes.
3. Une phrase comme `not good` est mal capturée en unigrammes purs: `good` peut dominer et conduire à une confusion si la négation n’est pas modélisée explicitement.

## 6. N-grams

Les n-grams ajoutent une information partielle sur l’ordre local des mots.

### Questions
- Que gagne-t-on avec les bigrammes ?
- Pourquoi les trigrammes sont-ils souvent plus rares ?
- Quel compromis semble le plus pertinent entre précision et complexité ?

### Réponses
- Les bigrammes capturent des expressions clés (`not good`, `very bad`, `highly recommend`) absentes de BoW unigramme.
- Les trigrammes sont plus rares car le nombre de combinaisons explose et beaucoup n’apparaissent que peu de fois.
- Un compromis efficace est souvent **(1,2)** : meilleur signal contextuel local avec un coût encore raisonnable.

In [ ]:
ngram_model = Pipeline([
    ("vect", CountVectorizer(max_features=15000, ngram_range=(1, 2))),
    ("clf", LogisticRegression(max_iter=1000))
])

t0 = time.time()
ngram_model.fit(X_train, y_train)
time_ngram = time.time() - t0

y_pred_ng = ngram_model.predict(X_test)
acc_ng = accuracy_score(y_test, y_pred_ng)
f1_ng = f1_score(y_test, y_pred_ng, average="macro")

print("Accuracy:", round(acc_ng, 4))
print("F1 macro:", round(f1_ng, 4))
print("Temps (s):", round(time_ngram, 2))

### À observer

Cherchez des expressions fréquentes comme :
- `not good`
- `really bad`
- `highly recommend`
- `waste of time`

Demandez-vous si le modèle les traite mieux que BoW.

## 7. TF-IDF

TF-IDF corrige la simple fréquence par une pondération liée à la rareté du terme dans le corpus.

### Questions
- Quels mots sont naturellement pénalisés ?
- Pourquoi TF-IDF est-il souvent supérieur à BoW ?
- Dans quels cas la différence peut-elle rester faible ?

### Réponses
- Les mots très fréquents dans tout le corpus (peu discriminants) sont pénalisés.
- TF-IDF améliore le rapport signal/bruit en rehaussant les termes informatifs pour la tâche.
- L’écart peut rester faible si BoW capture déjà l’essentiel des indices de polarité et si les classes sont bien séparées lexicalement.

In [ ]:
tfidf_model = Pipeline([
    ("vect", TfidfVectorizer(max_features=15000, ngram_range=(1, 2))),
    ("clf", LogisticRegression(max_iter=1000))
])

t0 = time.time()
tfidf_model.fit(X_train, y_train)
time_tfidf = time.time() - t0

y_pred_tfidf = tfidf_model.predict(X_test)
acc_tfidf = accuracy_score(y_test, y_pred_tfidf)
f1_tfidf = f1_score(y_test, y_pred_tfidf, average="macro")

print("Accuracy:", round(acc_tfidf, 4))
print("F1 macro:", round(f1_tfidf, 4))
print("Temps (s):", round(time_tfidf, 2))

### Question d’analyse

TF-IDF n’est pas meilleur parce qu’il “comprend” le texte au sens sémantique profond; il reste une méthode lexicale.

Son gain provient surtout de la **pondération** :
- les mots très fréquents et peu discriminants sont pénalisés ;
- les termes plus informatifs pour la classe ont un poids relatif plus élevé.

Autrement dit, TF-IDF améliore le signal statistique plutôt que la compréhension contextuelle.

## 8. Comparaison des méthodes lexicales

À ce stade, comparez :
- Bag of Words ;
- N-grams ;
- TF-IDF.

### Questions de conclusion
- Quelle méthode offre le meilleur compromis ?
- La meilleure performance est-elle forcément la plus simple à interpréter ?
- Quelle méthode recommanderiez-vous dans un contexte de déploiement rapide ?

### Réponses
- En pratique sur IMDb, **TF-IDF + n-grams** offre souvent le meilleur compromis performance/stabilité.
- La meilleure performance n’est pas toujours la plus interprétable, mais les méthodes lexicales restent globalement lisibles (poids de mots, n-grams discriminants).
- Pour un déploiement rapide: **TF-IDF + Régression Logistique** (rapide, robuste, explicable, peu coûteux).

In [ ]:
resultats_lexicaux = pd.DataFrame([
    {"Méthode": "BoW", "Accuracy": acc_bow, "F1_macro": f1_bow, "Temps_s": time_bow},
    {"Méthode": "N-grams", "Accuracy": acc_ng, "F1_macro": f1_ng, "Temps_s": time_ngram},
    {"Méthode": "TF-IDF", "Accuracy": acc_tfidf, "F1_macro": f1_tfidf, "Temps_s": time_tfidf},
])

display(resultats_lexicaux.sort_values("Accuracy", ascending=False).round(4))

## 9. Word Embeddings — Word2Vec

Un embedding dense représente un mot dans un espace vectoriel continu.

### Questions
- Que signifie une proximité entre deux mots dans cet espace ?
- Pourquoi un embedding peut-il capturer de la similarité sémantique ?
- Que perd-on lorsqu’on résume un document par la moyenne de ses vecteurs de mots ?

### Réponses
- Une proximité traduit des contextes d’usage semblables, donc souvent une similarité sémantique/fonctionnelle.
- Word2Vec apprend à prédire le contexte: des mots apparaissant dans des contextes proches obtiennent des vecteurs proches.
- La moyenne perd l’ordre des mots, les interactions syntaxiques et des phénomènes clés comme la négation portée par la structure de phrase.

In [ ]:
from gensim.models import Word2Vec

sentences = [str(t).split() for t in df["text_clean"]]
w2v = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    seed=42
)

print("Taille vocab Word2Vec:", len(w2v.wv))

> **Visualiser l’espace vectoriel des mots en utilisant t-SNE**


In [ ]:
def document_vector(tokens, model, vector_size=100):
    vectors = [model.wv[w] for w in tokens if w in model.wv]
    if len(vectors) == 0:
        return np.zeros(vector_size)
    return np.mean(vectors, axis=0)

X_w2v = np.vstack([
    document_vector(str(t).split(), w2v, 100)
    for t in df["text_clean"]
])

# Visualisation t-SNE d'un sous-ensemble de mots fréquents
words = list(w2v.wv.index_to_key[:200])
word_vecs = np.array([w2v.wv[w] for w in words])
coords = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(word_vecs)

plt.figure(figsize=(9, 6))
plt.scatter(coords[:, 0], coords[:, 1], s=18, alpha=0.7)
for i, w in enumerate(words[:60]):
    plt.annotate(w, (coords[i, 0], coords[i, 1]), fontsize=8, alpha=0.8)
plt.title("Projection t-SNE des embeddings Word2Vec")
plt.show()

In [ ]:
# Classification sur les vecteurs Word2Vec
X_train_w2v, X_test_w2v, y_train_w2v, y_test_w2v = train_test_split(
    X_w2v, y, test_size=0.2, random_state=42, stratify=y
)

t0 = time.time()
clf_w2v = LogisticRegression(max_iter=1000)
clf_w2v.fit(X_train_w2v, y_train_w2v)
time_w2v = time.time() - t0

y_pred_w2v = clf_w2v.predict(X_test_w2v)
acc_w2v = accuracy_score(y_test_w2v, y_pred_w2v)
f1_w2v = f1_score(y_test_w2v, y_pred_w2v, average="macro")

print("Accuracy:", round(acc_w2v, 4))
print("F1 macro:", round(f1_w2v, 4))
print("Temps (s):", round(time_w2v, 2))

### Interprétation guidée

Comparaison Word2Vec vs TF-IDF :
- Word2Vec capte mieux la proximité sémantique entre mots (synonymie/proximité de contexte).
- La moyenne des vecteurs de mots simplifie trop le document: elle perd l’ordre, la syntaxe et la structure de négation.
- Les mots rares sont moins bien représentés si peu vus pendant l’apprentissage.
- Les performances peuvent rester proches, voire inférieures à TF-IDF, car le problème de sentiment dépend fortement d’indices lexicaux explicites que TF-IDF exploite très bien.

## 10. FastText

FastText enrichit Word2Vec en intégrant des sous-mots.

### Questions
- En quoi cela aide-t-il pour les mots rares ?
- Pourquoi FastText est-il souvent plus robuste aux fautes d’orthographe ?
- Dans quelles langues cette propriété est-elle particulièrement utile ?

### Réponses
- Les mots rares profitent des n-grammes de caractères partagés avec d’autres mots fréquents.
- Une faute légère conserve souvent une partie des sous-mots, donc le vecteur reste exploitable.
- Cette propriété est très utile pour les langues morphologiquement riches (ex. arabe, turc, finnois) et dans les corpus bruités (réseaux sociaux).

In [ ]:
from gensim.models import FastText

ft = FastText(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    seed=42
)

def document_vector_ft(tokens, model, vector_size=100):
    vectors = [model.wv[w] for w in tokens if w in model.wv]
    if len(vectors) == 0:
        return np.zeros(vector_size)
    return np.mean(vectors, axis=0)

X_ft = np.vstack([
    document_vector_ft(str(t).split(), ft, 100)
    for t in df["text_clean"]
])

X_train_ft, X_test_ft, y_train_ft, y_test_ft = train_test_split(
    X_ft, y, test_size=0.2, random_state=42, stratify=y
)

t0 = time.time()
clf_ft = LogisticRegression(max_iter=1000)
clf_ft.fit(X_train_ft, y_train_ft)
time_ft = time.time() - t0

y_pred_ft = clf_ft.predict(X_test_ft)
acc_ft = accuracy_score(y_test_ft, y_pred_ft)
f1_ft = f1_score(y_test_ft, y_pred_ft, average="macro")

print("Accuracy:", round(acc_ft, 4))
print("F1 macro:", round(f1_ft, 4))
print("Temps (s):", round(time_ft, 2))

### Comparaison attendue

FastText n’est pas forcément plus performant parce qu’il “comprend mieux les émotions” que Word2Vec. Le gain vient surtout de la modélisation en **sous-mots** :
- meilleure gestion des formes rares, dérivées et fautives ;
- meilleure robustesse morphologique ;
- meilleure couverture lexicale globale.

Sur un corpus comme IMDb (anglais relativement propre), le gain peut être modéré mais reste souvent visible sur les cas OOV/variantes.

## 11. Embeddings contextuels — BERT

BERT produit des vecteurs dépendants du contexte. Un mot n’a donc plus un seul vecteur fixe.

### Questions
- Pourquoi la négation est-elle mieux traitée ?
- Pourquoi BERT distingue-t-il mieux les contextes ?
- Qu’apporte le contexte bidirectionnel par rapport à un embedding statique ?

### Réponses
- La négation est mieux traitée car BERT encode les interactions entre tokens de la phrase entière (`not good` n’est pas réduit à la somme de `not` et `good`).
- BERT distingue mieux les contextes via l’attention, qui pondère les mots pertinents selon l’énoncé complet.
- Le contexte bidirectionnel permet d’utiliser simultanément le voisinage gauche et droit du mot, alors qu’un embedding statique assigne un seul vecteur quel que soit le sens local.

In [ ]:
from transformers import AutoTokenizer, AutoModel

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name)
print("Modèle chargé:", model_name)

In [ ]:
def bert_embeddings(texts, tokenizer, model, max_length=256, batch_size=16):
    import torch
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    all_emb = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            out = model(**enc)

        cls_emb = out.last_hidden_state[:, 0, :].cpu().numpy()
        all_emb.append(cls_emb)

    return np.vstack(all_emb)

### Variante pédagogique

Pour limiter le coût de calcul, appliquez BERT sur un sous-échantillon raisonnable du corpus, puis comparez les tendances obtenues avec les méthodes précédentes.

## 12. Évaluation commune

Utilisez les mêmes métriques pour toutes les méthodes testées.

### Métriques recommandées
- accuracy ;
- F1 macro ;
- matrice de confusion.

### Questions
- Quelle métrique reste la plus informative si les classes sont déséquilibrées ?
- Que révèle la matrice de confusion sur les types d’erreurs ?

### Réponses
- En cas de déséquilibre, **F1 macro** est généralement plus informative que l’accuracy, car elle considère séparément les performances par classe.
- La matrice de confusion montre si le modèle confond surtout les positifs en négatifs (faux négatifs) ou l’inverse (faux positifs), et permet d’identifier un biais de décision.

In [ ]:
def evaluer(y_true, y_pred, nom="Modèle"):
    print(f"--- {nom} ---")
    print("Accuracy :", round(accuracy_score(y_true, y_pred), 4))
    print("F1 macro :", round(f1_score(y_true, y_pred, average="macro"), 4))
    print(classification_report(y_true, y_pred))

# Évaluation commune
evaluer(y_test, y_pred_bow, "BoW")
evaluer(y_test, y_pred_ng, "N-grams")
evaluer(y_test, y_pred_tfidf, "TF-IDF")
evaluer(y_test_w2v, y_pred_w2v, "Word2Vec")
evaluer(y_test_ft, y_pred_ft, "FastText")
evaluer(y_test_bert, y_pred_bert, "BERT")

In [ ]:
# BERT sur sous-échantillon pour limiter le coût
bert_df = df.sample(n=3000, random_state=42).reset_index(drop=True)
X_bert = bert_df["text_clean"].tolist()
y_bert = bert_df["label"].values

X_train_bert, X_test_bert, y_train_bert, y_test_bert = train_test_split(
    X_bert, y_bert, test_size=0.2, random_state=42, stratify=y_bert
)

t0 = time.time()
X_train_bert_emb = bert_embeddings(X_train_bert, tokenizer, bert_model, max_length=256, batch_size=16)
X_test_bert_emb = bert_embeddings(X_test_bert, tokenizer, bert_model, max_length=256, batch_size=16)

clf_bert = LogisticRegression(max_iter=1000)
clf_bert.fit(X_train_bert_emb, y_train_bert)
y_pred_bert = clf_bert.predict(X_test_bert_emb)
time_bert = time.time() - t0

acc_bert = accuracy_score(y_test_bert, y_pred_bert)
f1_bert = f1_score(y_test_bert, y_pred_bert, average="macro")

print("BERT + LR")
print("Accuracy:", round(acc_bert, 4))
print("F1 macro:", round(f1_bert, 4))
print("Temps (s):", round(time_bert, 2))

# Matrice de confusion (BERT)
cm = confusion_matrix(y_test_bert, y_pred_bert)
plt.figure(figsize=(4, 4))
plt.imshow(cm, interpolation="nearest", cmap="Blues")
plt.title("Matrice de confusion - BERT")
plt.colorbar()
plt.xticks([0, 1], ["Négatif", "Positif"])
plt.yticks([0, 1], ["Négatif", "Positif"])
plt.xlabel("Prédit")
plt.ylabel("Réel")
plt.show()

## 13. Analyse des erreurs

L’analyse des erreurs est une étape essentielle.

### À faire
Repérez quelques exemples :
- faux positifs ;
- faux négatifs ;
- cas ambigus ;
- textes contenant de la négation ;
- textes très courts ou très longs.

### Questions
- Les erreurs sont-elles aléatoires ?
- Les méthodes lexicales échouent-elles sur les mêmes cas que les embeddings ?
- BERT corrige-t-il vraiment les ambiguïtés observées plus tôt ?

### Réponses
- Les erreurs ne sont pas totalement aléatoires: elles se concentrent souvent sur les avis ambigus, ironiques, contrastés ou fortement négatifs avec vocabulaire positif local.
- Les méthodes lexicales échouent surtout quand la polarité dépend du contexte global; les embeddings gèrent mieux certains de ces cas.
- BERT corrige généralement une partie des ambiguïtés (négation, dépendances longues), mais pas toutes, surtout sur exemples très bruités ou ironiques.

In [ ]:
# Analyse d'erreurs sur TF-IDF (méthode lexicale de référence)
err_df = pd.DataFrame({
    "text": X_test.reset_index(drop=True),
    "true": y_test.reset_index(drop=True),
    "pred_tfidf": y_pred_tfidf
})

fp = err_df[(err_df["true"] == 0) & (err_df["pred_tfidf"] == 1)]
fn = err_df[(err_df["true"] == 1) & (err_df["pred_tfidf"] == 0)]

print("Faux positifs:", len(fp), "| Faux négatifs:", len(fn))
print("\nExemples faux positifs:")
display(fp.head(3))
print("\nExemples faux négatifs:")
display(fn.head(3))

## 14. Tableau comparatif final

Complétez un tableau récapitulatif avec vos résultats expérimentaux.

### Colonnes conseillées
- Méthode
- Type de représentation
- Accuracy
- F1 macro
- Temps d’exécution
- Avantages
- Limites

In [ ]:
resultats = pd.DataFrame([
    {
        "Méthode": "BoW",
        "Type": "Lexicale",
        "Accuracy": acc_bow,
        "F1_macro": f1_bow,
        "Temps": time_bow,
        "Avantage": "Simple et interprétable",
        "Limite": "Ignore l'ordre des mots"
    },
    {
        "Méthode": "N-grams",
        "Type": "Lexicale locale",
        "Accuracy": acc_ng,
        "F1_macro": f1_ng,
        "Temps": time_ngram,
        "Avantage": "Capture des expressions locales",
        "Limite": "Dimension plus grande"
    },
    {
        "Méthode": "TF-IDF",
        "Type": "Lexicale pondérée",
        "Accuracy": acc_tfidf,
        "F1_macro": f1_tfidf,
        "Temps": time_tfidf,
        "Avantage": "Bon compromis perf/coût",
        "Limite": "Reste non contextuel"
    },
    {
        "Méthode": "Word2Vec",
        "Type": "Embedding statique",
        "Accuracy": acc_w2v,
        "F1_macro": f1_w2v,
        "Temps": time_w2v,
        "Avantage": "Similarité sémantique",
        "Limite": "Perte d'info par moyenne"
    },
    {
        "Méthode": "FastText",
        "Type": "Embedding sous-mots",
        "Accuracy": acc_ft,
        "F1_macro": f1_ft,
        "Temps": time_ft,
        "Avantage": "Robuste aux mots rares/OOV",
        "Limite": "Toujours statique"
    },
    {
        "Méthode": "BERT",
        "Type": "Embedding contextuel",
        "Accuracy": acc_bert,
        "F1_macro": f1_bert,
        "Temps": time_bert,
        "Avantage": "Meilleure compréhension contextuelle",
        "Limite": "Coût de calcul élevé"
    },
])

display(resultats.sort_values("Accuracy", ascending=False).round(4))

## 15. Questions de synthèse

### Synthèse

- **Méthode la plus adaptée au corpus** : pour IMDb, les méthodes lexicales avancées (TF-IDF + bigrammes) donnent déjà un excellent niveau. BERT peut dépasser ces performances mais avec un coût nettement supérieur.
- **Interprétabilité** : la méthode la plus performante n’est pas toujours la plus interprétable. Les modèles lexicaux sont faciles à expliquer (poids des mots), alors que BERT est plus opaque.
- **Compromis recommandé** :
  - si priorité au coût/rapidité/explicabilité: **TF-IDF + Régression Logistique** ;
  - si priorité à la performance maximale: **BERT** (ou fine-tuning Transformer).
- **Effet de plus de données** : avec davantage de données, les approches contextuelles ont généralement plus de marge de progression, donc l’écart en faveur de BERT peut s’accentuer.